# Probing in Code LLMs - Index/Key with Multiple Variable Perturbations

| Strategy | Description |
|----------|-------------|
| `baseline` | Original variable names — control group |
| `random_nouns` | Replace with everyday English nouns (apple, drum, leaf…) |
| `single_chars` | Replace with single lowercase letters cycling a→z |
| `all_same` | Replace every variable with the same name `x` |
| `numeric_vars` | Replace with programmer-style names v1, v2, v3… |
| `misleading` | Index/key vars get accumulator-sounding names; non-index vars get index-sounding names |

The central question: **does Qwen2.5-1.5B detect index/key role from syntactic position,
or from surface variable names?**

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from collections import defaultdict
import re
# need the AST to properly identify subscript expressions
import ast
import random
import string
import os
import json
import warnings
warnings.filterwarnings('ignore')  # suppress the sklearn/tqdm noise — gets annoying fast

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # use GPU if it's there, otherwise fall back to CPU
print(f'Using device: {DEVICE}')

# where all the plots and CSVs will land
RESULTS_DIR = '../results/more_perturbated_variables'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

/Applications/Projects/Algoverse/Coding/algoverse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Results will be saved to: ../results/more_perturbated_variables


## Model Loading

In [2]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

print(f'Loading tokenizer and model: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)  # needed for Qwen — it has custom model code

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # Qwen has no separate pad token, eos works fine here

model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True, trust_remote_code=True)  # needed for Qwen — it has custom model code
model.eval()  # inference only, no gradients needed
model.to(DEVICE)  # move weights to GPU if available

# grab these once — reuse later
NUM_LAYERS  = model.config.num_hidden_layers
HIDDEN_SIZE = model.config.hidden_size
print(f'Layers: {NUM_LAYERS}, Hidden size: {HIDDEN_SIZE}')

_probe_ids = tokenizer('x', return_tensors='pt')['input_ids'][0].tolist()
_raw_len   = len(tokenizer.tokenize('x'))
LEADING_SPECIAL = len(_probe_ids) - _raw_len  # BERT has [CLS] so offset=1; Qwen has none so offset=0
print(f'Leading special tokens: {LEADING_SPECIAL}')

Loading tokenizer and model: Qwen/Qwen2.5-1.5B ...


Loading weights: 100%|██████████| 338/338 [00:00<00:00, 5308.31it/s]

Layers: 28, Hidden size: 1536
Leading special tokens: 0


## Dataset Loading

In [3]:
# adjust this path if the data is somewhere else
XLCOST_ROOT  = '/Applications/Projects/Algoverse/Coding/XLCoST_data'
NL2CODE_PROG = os.path.join(XLCOST_ROOT, 'retrieval', 'nl2code_search', 'program_level')


def reconstruct_code(tokens):  # XLCoST stores code as token lists with indent markers
    """Reconstruct code from XLCoST token list."""
    indent_level, lines, current_line = 0, [], []
    for tok in tokens:
        if tok == 'NEW_LINE':  # XLCoST encodes newlines as explicit tokens
            lines.append('    ' * indent_level + ' '.join(current_line))
            current_line = []
        elif tok == 'INDENT':  # indentation is tracked via these special tokens
            indent_level += 1
        elif tok == 'DEDENT':
            indent_level = max(0, indent_level - 1)
        else:
            current_line.append(tok)
    if current_line:
        lines.append('    ' * indent_level + ' '.join(current_line))
    return '\n'.join(lines).strip()


def load_xlcost_programs(language='Python', split='train', max_programs=300):  # cap how many programs we load — enough for a good sample
    """Load raw (unmodified) programs from XLCoST."""
    path = os.path.join(NL2CODE_PROG, language, f'{split}.jsonl')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Not found: {path}')
    programs = []
    with open(path) as f:
        for line in f:
            if len(programs) >= max_programs:  # cap how many programs we load — enough for a good sample
                break
            rec  = json.loads(line.strip())  # each line in the .jsonl is one program
            code = reconstruct_code(rec['code_tokens'])  # XLCoST stores code as token lists with indent markers
            if code:
                programs.append(code)
    return programs


print('Loading Python programs from XLCoST...')
RAW_PROGRAMS = load_xlcost_programs('Python', split='train', max_programs=500)  # cap how many programs we load — enough for a good sample
print(f'Loaded {len(RAW_PROGRAMS)} Python programs')

Loading Python programs from XLCoST...
Loaded 500 Python programs


## Core Utilities — AST Labeling & Tokenization

In [4]:
import keyword
import builtins

PYTHON_PROTECTED = (  # don't rename keywords or builtins — code would break
    set(keyword.kwlist)
    | set(dir(builtins))
    | {'self', 'cls', 'None', 'True', 'False'}
)


def get_index_key_names(code):
    """Return variable names used as array indices or dict keys (Python AST)."""
    indices = set()
    try:
        tree = ast.parse(code)  # AST is more reliable than regex for finding subscripts
    except SyntaxError:
        return indices
    for node in ast.walk(tree):
        if isinstance(node, ast.Subscript):  # subscript nodes cover both arr[i] and d[key]
            slc = node.slice
            if isinstance(slc, ast.Index):   # Python < 3.9
                slc = slc.value
            if isinstance(slc, ast.Name):  # only care about variable names, not literals like arr[0]
                indices.add(slc.id)
    return indices


def collect_all_identifiers(code):
    """Return all identifier names in a Python program (AST-based)."""
    names = set()
    try:
        tree = ast.parse(code)  # AST is more reliable than regex for finding subscripts
        for node in ast.walk(tree):
            if isinstance(node, ast.Name):  # only care about variable names, not literals like arr[0]
                names.add(node.id)
            elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                names.add(node.name)
            elif isinstance(node, ast.arg):
                names.add(node.arg)
    except SyntaxError:
        names = set(re.findall(r'\b[a-zA-Z_]\w*\b', code))
    return names


def label_tokens(code, target_names):
    """
    Tokenize code and label each token 1 if it matches a target name, else 0.
    Handles Qwen bracket-merging quirk ([i → i).
    """
    if not target_names:  # nothing to label if no index vars were found
        return [], []
    tokens = tokenizer.tokenize(code)
    labels = []
    for tok in tokens:
        is_continuation = tok.startswith('##')  # BERT wordpiece continuation tokens — skip these
        clean           = tok.lstrip('Ġ▁Ā').lstrip('##')  # GPT-style tokenizers prefix spaces with Ġ
        clean_inner     = clean.strip('[]().,;:')
        matched = (
            (not is_continuation)
            and (clean in target_names or clean_inner in target_names)
        )
        labels.append(1 if matched else 0)
    return tokens, labels


def apply_rename_map(code, rename_map):  # whole-word substitution, longest names first to avoid partial matches
    """Apply whole-word substitutions (longest first) to code."""
    if not rename_map:
        return code
    for orig in sorted(rename_map, key=len, reverse=True):  # replace longer names first — avoids 'i' replacing inside 'index'
        code = re.sub(r'\b' + re.escape(orig) + r'\b', rename_map[orig], code)  # escape in case a variable name has special regex chars
    return code


print('Utilities loaded.')

Utilities loaded.


## Perturbation Strategies

Each strategy returns a `(renamed_code, new_index_key_names)` pair.  
The probe labels are derived from `new_index_key_names` in the *transformed* code.

In [5]:
# ── Name pools ────────────────────────────────────────────────────────────────

RANDOM_NOUNS = sorted(set([
    'apple', 'bag', 'ball', 'basil', 'bear', 'bed', 'belt', 'birch', 'bird', 'bolt',
    'bowl', 'box', 'broom', 'brush', 'cage', 'cart', 'cat', 'cedar', 'clip', 'coat',
    'coin', 'comb', 'cord', 'crow', 'cube', 'cup', 'deer', 'desk', 'dial', 'dish',
    'dog', 'dome', 'drop', 'drum', 'duck', 'dust', 'edge', 'fern', 'fish', 'flame',
    'fork', 'fox', 'frog', 'gate', 'gear', 'gem', 'glove', 'glow', 'grain', 'grid',
    'hat', 'hawk', 'heap', 'hill', 'hole', 'hook', 'hose', 'hull', 'jar', 'kelp',
    'knob', 'knot', 'lamb', 'lamp', 'latch', 'leaf', 'lens', 'lion', 'lobe', 'lock',
    'loop', 'lump', 'maple', 'mark', 'mast', 'mesh', 'mint', 'mist', 'mold', 'mole',
    'moss', 'nail', 'net', 'newt', 'node', 'oak', 'orb', 'palm', 'path', 'peak',
    'pen', 'pile', 'pine', 'pipe', 'plank', 'plate', 'plug', 'pool', 'port', 'pot',
    'puma', 'ramp', 'reed', 'reef', 'ring', 'rod', 'root', 'rope', 'rose', 'rug',
    'sage', 'sand', 'seal', 'seed', 'slab', 'slot', 'stem', 'tree', 'vine', 'wolf',
]))

# Names that LOOK like index/loop variables (conventional)
INDEX_LOOKING  = ['i', 'j', 'k', 'n', 'm', 'idx', 'pos', 'key', 'ptr',
                  'cur', 'row', 'col', 'x', 'y', 'p', 'q', 'r', 't', 'u']

# Names that LOOK like accumulators / non-index (conventional)
NOINDEX_LOOKING = ['total', 'count', 'result', 'output', 'answer', 'value',
                   'amount', 'data', 'temp', 'buf', 'res', 'ret', 'score',
                   'maximum', 'minimum', 'current', 'previous', 'running',
                   'accumulator', 'product', 'difference']


def _renameable(code):
    """Return sorted list of renameable identifiers."""
    return sorted(n for n in collect_all_identifiers(code) if n not in PYTHON_PROTECTED)  # don't rename keywords or builtins — code would break


# ── 1. Random nouns ───────────────────────────────────────────────────────────
def perturb_random_nouns(code, seed=0):
    rng     = random.Random(seed)
    names   = _renameable(code)
    pool    = rng.sample(RANDOM_NOUNS, min(len(RANDOM_NOUNS), len(names)))  # sample without replacement so no two vars get the same new name
    if len(names) > len(pool):
        pool += rng.choices(RANDOM_NOUNS, k=len(names) - len(pool))
    rmap    = dict(zip(names, pool))
    new_code = apply_rename_map(code, rmap)  # whole-word substitution, longest names first to avoid partial matches
    new_idx  = get_index_key_names(new_code)   # AST on renamed code
    return new_code, new_idx


# ── 2. Single characters (a, b, c … z, aa, ab …) ────────────────────────────
def _single_char_pool(n):
    """Generate n single-char (or short) unique identifiers cycling a..z."""
    pool = list(string.ascii_lowercase)
    # extend if more than 26 vars
    suffix = 0
    while len(pool) < n:
        for c in string.ascii_lowercase:
            pool.append(c + str(suffix))
        suffix += 1
    return pool[:n]

def perturb_single_chars(code, seed=0):
    rng   = random.Random(seed)
    names = _renameable(code)
    pool  = _single_char_pool(len(names))
    rng.shuffle(pool)                         # shuffle so assignment is random
    rmap     = dict(zip(names, pool))
    new_code = apply_rename_map(code, rmap)  # whole-word substitution, longest names first to avoid partial matches
    new_idx  = get_index_key_names(new_code)
    return new_code, new_idx


# ── 3. All same name ('x') ───────────────────────────────────────────────────
def perturb_all_same(code, seed=0):
    names    = _renameable(code)
    rmap     = {n: 'x' for n in names}
    new_code = apply_rename_map(code, rmap)  # whole-word substitution, longest names first to avoid partial matches
    new_idx  = get_index_key_names(new_code)
    return new_code, new_idx


# ── 4. Numeric names (v1, v2, v3 …) ─────────────────────────────────────────
def perturb_numeric_vars(code, seed=0):
    rng      = random.Random(seed)
    names    = _renameable(code)
    indices  = list(range(1, len(names) + 1))
    rng.shuffle(indices)
    rmap     = {n: f'v{idx}' for n, idx in zip(names, indices)}
    new_code = apply_rename_map(code, rmap)  # whole-word substitution, longest names first to avoid partial matches
    new_idx  = get_index_key_names(new_code)
    return new_code, new_idx


# ── 5. Misleading names ───────────────────────────────────────────────────────
# Index/key vars  → accumulator-sounding names (total, count, result …)
# Non-index vars  → index-sounding names       (i, j, k, idx, pos …)
def perturb_misleading(code, seed=0):
    rng      = random.Random(seed)
    all_names = _renameable(code)
    idx_names = get_index_key_names(code)     # original index/key vars

    index_vars  = sorted(n for n in all_names if n in idx_names)
    normal_vars = sorted(n for n in all_names if n not in idx_names)

    # Extend pools if needed
    noindex_pool = NOINDEX_LOOKING * (len(index_vars) // len(NOINDEX_LOOKING) + 2)
    rng.shuffle(noindex_pool)
    index_pool   = INDEX_LOOKING * (len(normal_vars) // len(INDEX_LOOKING) + 2)
    rng.shuffle(index_pool)

    rmap = {}
    for var, new_name in zip(index_vars, noindex_pool):
        rmap[var] = new_name          # index var → looks like accumulator
    for var, new_name in zip(normal_vars, index_pool):
        rmap[var] = new_name          # normal var → looks like index

    new_code = apply_rename_map(code, rmap)  # whole-word substitution, longest names first to avoid partial matches
    new_idx  = get_index_key_names(new_code)   # AST detects roles in renamed code
    return new_code, new_idx


# ── Strategy registry ─────────────────────────────────────────────────────────
PERTURBATIONS = {
    'baseline':     None,                  # no transformation — handled separately
    'random_nouns': perturb_random_nouns,
    'single_chars': perturb_single_chars,
    'all_same':     perturb_all_same,
    'numeric_vars': perturb_numeric_vars,
    'misleading':   perturb_misleading,
}

print('Perturbation strategies defined:', list(PERTURBATIONS.keys()))

# ── Quick sanity check ────────────────────────────────────────────────────────
demo = 'total = 0\nfor i in range(n):\n    total += arr[i]\nd[key] = d.get(key, 0) + 1'
print('\nOriginal:    ', get_index_key_names(demo))
for name, fn in PERTURBATIONS.items():
    if fn is None:
        print(f'{name:15s}  (no change)')
        continue
    new_code, new_idx = fn(demo, seed=42)
    print(f'{name:15s}  {new_idx}  | code: {new_code[:60]!r}')

Perturbation strategies defined: ['baseline', 'random_nouns', 'single_chars', 'all_same', 'numeric_vars', 'misleading']

Original:     {'key', 'i'}
baseline         (no change)
random_nouns     {'basil', 'plank'}  | code: 'dome = 0\nfor basil in range(dust):\n    dome += nail[basil]\nc'
single_chars     {'e', 'c'}  | code: 'f = 0\nfor c in range(a):\n    f += b[c]\nb[e] = b.get(e, 0) + '
all_same         {'x'}  | code: 'x = 0\nfor x in range(x):\n    x += x[x]\nx[x] = x.get(x, 0) + '
numeric_vars     {'v3', 'v5'}  | code: 'v6 = 0\nfor v3 in range(v1):\n    v6 += v4[v3]\nv2[v5] = v2.get'
misleading       {'buf', 'output'}  | code: 'buf = 0\nfor buf in range(y):\n    buf += p[buf]\nq[output] = q'


## Dataset Builder, Hidden State Extractor, Probe Trainer

In [6]:
# progress bars — helpful when extraction takes a while
from tqdm.auto import tqdm

# anything longer than this gets truncated
MAX_SEQ_LEN = 512


def build_dataset(programs, perturb_fn=None, role_name='index_key'):
    """
    Apply perturbation (or none) to each program and build labeled token dataset.
    Returns list of dicts with 'code', 'tokens', 'labels', 'target_names'.
    """
    dataset, skipped = [], 0
    for idx, code in enumerate(programs):
        if perturb_fn is None:
            new_code  = code
            new_idx   = get_index_key_names(code)
        else:
            new_code, new_idx = perturb_fn(code, seed=idx)

        if not new_idx:
            skipped += 1
            continue

        tokens, labels = label_tokens(new_code, new_idx)
        if not tokens or sum(labels) == 0:  # skip programs where none of the index vars survived tokenization
            skipped += 1
            continue

        dataset.append({
            'code': new_code, 'tokens': tokens, 'labels': labels,
            'target_names': new_idx, 'role': role_name,
        })

    total_pos = sum(sum(d['labels']) for d in dataset)
    total_tok = sum(len(d['tokens']) for d in dataset)
    pct = 100 * total_pos / total_tok if total_tok else 0
    print(f'[{role_name:18s}]  {len(dataset):3d} programs  |  '
          f'{total_pos:,} pos / {total_tok:,} total ({pct:.1f}%)  |  {skipped} skipped')
    return dataset


def extract_hidden_states(dataset):
    """Extract per-layer hidden states for all tokens in the dataset."""
    all_hidden     = defaultdict(list)
    all_labels     = []
    all_token_strs = []

    with torch.no_grad():  # saves memory — we're not doing any backprop
        for sample in tqdm(dataset, desc='  Extracting hidden states', leave=False):
            code, labels = sample['code'], sample['labels']

            encoding = tokenizer(
                code, return_tensors='pt',
                truncation=True, max_length=MAX_SEQ_LEN, padding=False,  # cap at 512 — longer programs just get cut off
            ).to(DEVICE)

            n_content        = encoding['input_ids'].shape[1] - LEADING_SPECIAL  # strip the leading special token(s) before aligning labels
            tokens_no_special = tokenizer.tokenize(code)
            labels_trunc     = labels[:n_content]
            tokens_trunc     = tokens_no_special[:n_content]

            outputs      = model(**encoding)
            hidden_states = outputs.hidden_states

            start, end = LEADING_SPECIAL, LEADING_SPECIAL + n_content
            for layer_idx, layer_hs in enumerate(hidden_states):
                content_hs = layer_hs[0, start:end, :].float().cpu().numpy()  # Qwen uses bfloat16 by default — numpy can't handle it, cast first
                all_hidden[layer_idx].extend(content_hs)

            all_labels.extend(labels_trunc)
            all_token_strs.extend(tokens_trunc)

    for layer_idx in all_hidden:
        all_hidden[layer_idx] = np.stack(all_hidden[layer_idx])  # convert list of vectors to a proper 2D array

    return all_hidden, np.array(all_labels), all_token_strs


def train_probes(hidden_states_by_layer, labels, test_size=0.2, random_state=42):  # 80/20 split
    """Train one logistic regression probe per layer."""
    n = len(labels)
    train_idx, test_idx = train_test_split(
        np.arange(n), test_size=test_size, random_state=random_state, stratify=labels  # make sure both train and test have the same class ratio
    )
    results = {}
    for layer_idx in tqdm(sorted(hidden_states_by_layer.keys()), desc='  Training probes', leave=False):
        X = hidden_states_by_layer[layer_idx]
        clf = LogisticRegression(max_iter=1000, class_weight='balanced',  # index tokens are ~6% of all tokens — need to reweight or probe ignores them
                                  random_state=random_state, C=1.0)  # standard regularization, didn't tune this
        clf.fit(X[train_idx], labels[train_idx])
        y_pred_train = clf.predict(X[train_idx])
        y_pred_test  = clf.predict(X[test_idx])
        results[layer_idx] = {
            'train_acc': accuracy_score(labels[train_idx], y_pred_train),
            'test_acc':  accuracy_score(labels[test_idx],  y_pred_test),
            'train_f1':  f1_score(labels[train_idx], y_pred_train, average='macro'),  # macro F1 is fairer given the class imbalance
            'test_f1':   f1_score(labels[test_idx],  y_pred_test,  average='macro'),  # macro F1 is fairer given the class imbalance
            'probe':     clf,
            'test_idx':  test_idx,
        }
    return results


print('Pipeline functions defined.')

Pipeline functions defined.


: 

## Run All Perturbations

For each strategy: build dataset → extract hidden states → train probes.  
All results are stored in `all_results` keyed by strategy name.

In [7]:
all_results  = {}   # strategy → probe results dict
all_hidden   = {}   # strategy → hidden state dict
all_labels   = {}   # strategy → labels array

for strategy_name, perturb_fn in PERTURBATIONS.items():
    print(f'\n{"="*60}')
    print(f'  Strategy: {strategy_name}')
    print(f'{"="*60}')

    # 1. Build labeled dataset
    dataset = build_dataset(
        RAW_PROGRAMS,
        perturb_fn=perturb_fn,
        role_name=f'index_key_{strategy_name}'
    )
    if not dataset:
        print(f'  No valid data — skipping.')
        continue

    n_neg = sum(1 for d in dataset for l in d['labels'] if l == 0)
    n_pos = sum(1 for d in dataset for l in d['labels'] if l == 1)
    print(f'  Class balance: {n_neg:,} neg / {n_pos:,} pos  '
          f'(naive baseline: {100*n_neg/(n_neg+n_pos):.1f}%)')

    # 2. Extract hidden states
    h, lbs, _ = extract_hidden_states(dataset)
    all_hidden[strategy_name] = h
    all_labels[strategy_name] = lbs
    print(f'  Tokens extracted: {len(lbs):,}')

    # 3. Train probes
    results = train_probes(h, lbs)
    all_results[strategy_name] = results

    best_layer = max(results, key=lambda l: results[l]['test_f1'])
    print(f'  Best layer: {best_layer}  '
          f'test acc={results[best_layer]["test_acc"]:.3f}  '
          f'test F1={results[best_layer]["test_f1"]:.3f}')

print('\nAll strategies complete:', list(all_results.keys()))


  Strategy: baseline
[index_key_baseline]  255 programs  |  3,747 pos / 64,536 total (5.8%)  |  245 skipped
  Class balance: 60,789 neg / 3,747 pos  (naive baseline: 94.2%)


  Tokens extracted: 63,851


  Best layer: 8  test acc=0.991  test F1=0.959

  Strategy: random_nouns
[index_key_random_nouns]  251 programs  |  3,652 pos / 61,622 total (5.9%)  |  249 skipped
  Class balance: 57,970 neg / 3,652 pos  (naive baseline: 94.1%)


  Tokens extracted: 61,148


  Best layer: 13  test acc=0.985  test F1=0.934

  Strategy: single_chars
[index_key_single_chars]  255 programs  |  4,458 pos / 61,631 total (7.2%)  |  245 skipped
  Class balance: 57,173 neg / 4,458 pos  (naive baseline: 92.8%)


...

  Best layer: 9  test acc=1.000  test F1=1.000

  Strategy: numeric_vars
[index_key_numeric_vars]    1 programs  |  9 pos / 566 total (1.6%)  |  499 skipped
  Class balance: 557 neg / 9 pos  (naive baseline: 98.4%)


  Tokens extracted: 512


  Best layer: 0  test acc=1.000  test F1=1.000

  Strategy: misleading


[index_key_misleading]  255 programs  |  4,295 pos / 61,585 total (7.0%)  |  245 skipped
  Class balance: 57,290 neg / 4,295 pos  (naive baseline: 93.0%)


  Extracting hidden states:  56%|█████▌    | 143/255 [09:56<18:19,  9.82s/it]

## Per-Layer Tables

In [ ]:
layers = sorted(next(iter(all_results.values())).keys())

for strategy_name, results in all_results.items():
    print(f'\n── {strategy_name} ─────────────────────────────────────────')
    print(f'{"Layer":>7}  {"Train Acc":>10}  {"Test Acc":>10}  {"Train F1":>10}  {"Test F1":>10}')
    print('-' * 56)
    for l in layers:
        r = results[l]
        print(f'{l:>7}  {r["train_acc"]:>10.3f}  {r["test_acc"]:>10.3f}  '
              f'{r["train_f1"]:>10.3f}  {r["test_f1"]:>10.3f}')

## Visualization

In [ ]:
# ── Colour palette ────────────────────────────────────────────────────────────
COLORS = {
    'baseline':     '#FF5722',
    'random_nouns': '#2196F3',
    'single_chars': '#4CAF50',
    'all_same':     '#9C27B0',
    'numeric_vars': '#FF9800',
    'misleading':   '#F44336',
}
MARKERS = {
    'baseline':     'o',
    'random_nouns': 's',
    'single_chars': '^',
    'all_same':     'D',
    'numeric_vars': 'P',
    'misleading':   'X',
}
LINESTYLES = {
    'baseline':     '-',
    'random_nouns': '--',
    'single_chars': '-.',
    'all_same':     ':',
    'numeric_vars': (0, (3, 1, 1, 1)),
    'misleading':   (0, (5, 2)),
}
LABELS = {
    'baseline':     'Baseline (original names)',
    'random_nouns': 'Random nouns (apple, drum…)',
    'single_chars': 'Single chars (a, b, c…)',
    'all_same':     'All same name (x)',
    'numeric_vars': 'Numeric (v1, v2, v3…)',
    'misleading':   'Misleading (swapped names)',
}


# ── Plot 1: Test Accuracy ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

for strategy, results in all_results.items():
    accs = [results[l]['test_acc'] for l in layers]
    ax.plot(
        layers, accs,
        marker=MARKERS.get(strategy, 'o'),
        color=COLORS.get(strategy, 'gray'),
        linestyle=LINESTYLES.get(strategy, '-'),
        linewidth=2,
        markersize=5,
        label=LABELS.get(strategy, strategy),
    )

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance (0.5)')  # chance-level for a balanced binary probe
ax.set_xlabel('Layer Index  (0 = embedding)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title(
    f'Index/Key Probe — Test Accuracy Across All Perturbations\n{MODEL_NAME.split("/")[-1]}',
    fontsize=13
)
ax.set_ylim(0.3, 1.05)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/all_perturbations_accuracy.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: all_perturbations_accuracy.png')

In [ ]:
# ── Plot 2: Macro F1 ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

for strategy, results in all_results.items():
    f1s = [results[l]['test_f1'] for l in layers]
    ax.plot(
        layers, f1s,
        marker=MARKERS.get(strategy, 'o'),
        color=COLORS.get(strategy, 'gray'),
        linestyle=LINESTYLES.get(strategy, '-'),
        linewidth=2,
        markersize=5,
        label=LABELS.get(strategy, strategy),
    )

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance (0.5)')  # chance-level for a balanced binary probe
ax.set_xlabel('Layer Index  (0 = embedding)', fontsize=12)
ax.set_ylabel('Test Macro F1', fontsize=12)
ax.set_title(
    f'Index/Key Probe — Macro F1 Across All Perturbations\n{MODEL_NAME.split("/")[-1]}',
    fontsize=13
)
ax.set_ylim(0.3, 1.05)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/all_perturbations_f1.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: all_perturbations_f1.png')

In [ ]:
# ── Plot 3: Best-layer F1 bar chart ──────────────────────────────────────────
strategy_names = list(all_results.keys())
best_f1s       = [max(all_results[s][l]['test_f1'] for l in layers) for s in strategy_names]
best_layers    = [max(all_results[s], key=lambda l: all_results[s][l]['test_f1']) for s in strategy_names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    strategy_names, best_f1s,
    color=[COLORS.get(s, 'gray') for s in strategy_names],
    edgecolor='black', linewidth=0.6, width=0.6
)

# Annotate with value + best layer
for bar, f1, layer in zip(bars, best_f1s, best_layers):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.008,
        f'F1={f1:.3f}\n(L{layer})',
        ha='center', va='bottom', fontsize=9
    )

ax.set_xlabel('Perturbation Strategy', fontsize=12)
ax.set_ylabel('Best-Layer Test Macro F1', fontsize=12)
ax.set_title(
    f'Peak Probe F1 per Perturbation — {MODEL_NAME.split("/")[-1]}',
    fontsize=13
)
ax.set_ylim(0.0, 1.15)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='Chance')  # chance-level for a balanced binary probe
ax.tick_params(axis='x', rotation=15)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)  # slightly transparent so overlapping points are visible
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/best_layer_f1_bar.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: best_layer_f1_bar.png')

In [ ]:
# ── Plot 4: Heatmap — strategy × layer F1 ────────────────────────────────────
heatmap_data = np.array([
    [all_results[s][l]['test_f1'] for l in layers]
    for s in strategy_names
])

fig, ax = plt.subplots(figsize=(14, len(strategy_names) * 0.9 + 1.5))
im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlGn', vmin=0.5, vmax=1.0)

ax.set_xticks(range(len(layers)))
ax.set_xticklabels(layers, fontsize=8)
ax.set_yticks(range(len(strategy_names)))
ax.set_yticklabels([LABELS.get(s, s) for s in strategy_names], fontsize=10)
ax.set_xlabel('Layer Index', fontsize=12)
ax.set_title(
    f'Index/Key Probe Test F1 — Strategy × Layer Heatmap\n{MODEL_NAME.split("/")[-1]}',
    fontsize=13
)
plt.colorbar(im, ax=ax, label='Test Macro F1', fraction=0.02, pad=0.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/heatmap_strategy_x_layer.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
plt.show()
print('Saved: heatmap_strategy_x_layer.png')

In [ ]:
# ── Plot 5: Δ F1 vs baseline (degradation per perturbation) ──────────────────
if 'baseline' in all_results:
    baseline_f1 = np.array([all_results['baseline'][l]['test_f1'] for l in layers])

    fig, ax = plt.subplots(figsize=(12, 6))
    for strategy, results in all_results.items():
        if strategy == 'baseline':
            continue
        perturbed_f1 = np.array([results[l]['test_f1'] for l in layers])
        delta = perturbed_f1 - baseline_f1
        ax.plot(
            layers, delta,
            marker=MARKERS.get(strategy, 'o'),
            color=COLORS.get(strategy, 'gray'),
            linestyle=LINESTYLES.get(strategy, '-'),
            linewidth=2,
            markersize=5,
            label=LABELS.get(strategy, strategy),
        )

    ax.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.7, label='Baseline (Δ=0)')  # zero line for reference
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Δ Test Macro F1  (perturbation − baseline)', fontsize=12)
    ax.set_title(
        f'Probe F1 Degradation vs Baseline — All Perturbations\n{MODEL_NAME.split("/")[-1]}',
        fontsize=13
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/delta_f1_vs_baseline.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
    plt.show()
    print('Saved: delta_f1_vs_baseline.png')

## Best-Layer Classification Reports

In [ ]:
for strategy_name, results in all_results.items():
    best_layer = max(results, key=lambda l: results[l]['test_f1'])
    test_idx   = results[best_layer]['test_idx']
    clf        = results[best_layer]['probe']

    lbs = all_labels[strategy_name]
    h   = all_hidden[strategy_name]

    X_test = h[best_layer][test_idx]
    y_test = lbs[test_idx]
    y_pred = clf.predict(X_test)

    print(f'\n── {strategy_name}  (best layer: {best_layer}) ────────────────────────────────')
    print(classification_report(y_test, y_pred, target_names=['non-index', 'index_key']))

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f'  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
    print(f'  TPR={tp/(tp+fn):.3f}  FPR={fp/(fp+tn):.3f}  Precision={tp/(tp+fp):.3f}')

## Probe Weight Alignment — Cosine Similarity vs Baseline

Measures how similar each perturbation's probe direction is to the baseline probe.  
High similarity = the model uses the same latent direction regardless of surface names.

In [ ]:
from numpy.linalg import norm

if 'baseline' not in all_results:
    print('Baseline results not available — skipping cosine plot.')
else:
    fig, ax = plt.subplots(figsize=(12, 5))

    cosine_table = {}
    for strategy_name, results in all_results.items():
        if strategy_name == 'baseline':
            continue
        sims = []
        for l in layers:
            w_base   = all_results['baseline'][l]['probe'].coef_[0]
            w_perturb = results[l]['probe'].coef_[0]
            sims.append(np.dot(w_base, w_perturb) / (norm(w_base) * norm(w_perturb)))
        cosine_table[strategy_name] = sims

        ax.plot(
            layers, sims,
            marker=MARKERS.get(strategy_name, 'o'),
            color=COLORS.get(strategy_name, 'gray'),
            linestyle=LINESTYLES.get(strategy_name, '-'),
            linewidth=2,
            markersize=5,
            label=LABELS.get(strategy_name, strategy_name),
        )

    ax.axhline(1.0, color='green', linestyle=':', alpha=0.5, label='Perfect alignment (1.0)')
    ax.axhline(0.0, color='gray',  linestyle='--', alpha=0.4)  # zero line for reference
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Cosine Similarity with Baseline Probe', fontsize=12)
    ax.set_title(
        f'Probe Direction Alignment vs Baseline — All Perturbations\n{MODEL_NAME.split("/")[-1]}',
        fontsize=13
    )
    ax.set_ylim(-0.3, 1.15)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)  # slightly transparent so overlapping points are visible
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/cosine_vs_baseline.png', dpi=150, bbox_inches='tight')  # high enough for slides/paper without being huge
    plt.show()
    print('Saved: cosine_vs_baseline.png')

    # Summary table
    print(f'\n{"Strategy":18s}  ', end='')
    sample_layers = [0, 4, 8, 12, 16, 20, 24, 28]
    for l in sample_layers:
        if l in layers:
            print(f'  L{l:2d}', end='')
    print()
    print('-' * 70)
    for s, sims in cosine_table.items():
        print(f'{s:18s}  ', end='')
        for l in sample_layers:
            if l in layers:
                print(f'{sims[l]:6.3f}', end='')
        print()

## Summary Table

In [ ]:
print(f'\n{"Strategy":18s}  {"Best Layer":>10}  {"Best Acc":>9}  {"Best F1":>8}  '
      f'{"Δ Acc vs base":>14}  {"Δ F1 vs base":>13}')
print('-' * 82)

base_best_acc = max(all_results['baseline'][l]['test_acc'] for l in layers) if 'baseline' in all_results else None
base_best_f1  = max(all_results['baseline'][l]['test_f1']  for l in layers) if 'baseline' in all_results else None

for s, results in all_results.items():
    best_l   = max(results, key=lambda l: results[l]['test_f1'])
    best_acc = results[best_l]['test_acc']
    best_f1  = results[best_l]['test_f1']
    d_acc = f'{best_acc - base_best_acc:+.3f}' if base_best_acc and s != 'baseline' else '—'
    d_f1  = f'{best_f1  - base_best_f1 :+.3f}' if base_best_f1  and s != 'baseline' else '—'
    print(f'{s:18s}  {best_l:>10d}  {best_acc:>9.3f}  {best_f1:>8.3f}  {d_acc:>14}  {d_f1:>13}')

## Save Numeric Results to CSV

In [ ]:
import csv

# Save per-layer results for all strategies
csv_path = f'{RESULTS_DIR}/per_layer_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['strategy', 'layer', 'train_acc', 'test_acc', 'train_f1', 'test_f1'])
    for strategy, results in all_results.items():
        for l in layers:
            r = results[l]
            writer.writerow([strategy, l,
                             f'{r["train_acc"]:.4f}', f'{r["test_acc"]:.4f}',
                             f'{r["train_f1"]:.4f}',  f'{r["test_f1"]:.4f}'])
print(f'Saved: {csv_path}')

# Save summary (best layer per strategy)
summary_path = f'{RESULTS_DIR}/summary.csv'
with open(summary_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['strategy', 'best_layer', 'best_test_acc', 'best_test_f1',
                     'delta_acc_vs_baseline', 'delta_f1_vs_baseline'])
    base_best_acc = max(all_results['baseline'][l]['test_acc'] for l in layers) if 'baseline' in all_results else None
    base_best_f1  = max(all_results['baseline'][l]['test_f1']  for l in layers) if 'baseline' in all_results else None
    for s, results in all_results.items():
        best_l   = max(results, key=lambda l: results[l]['test_f1'])
        best_acc = results[best_l]['test_acc']
        best_f1  = results[best_l]['test_f1']
        d_acc = f'{best_acc - base_best_acc:+.4f}' if base_best_acc and s != 'baseline' else ''
        d_f1  = f'{best_f1  - base_best_f1 :+.4f}' if base_best_f1  and s != 'baseline' else ''
        writer.writerow([s, best_l, f'{best_acc:.4f}', f'{best_f1:.4f}', d_acc, d_f1])
print(f'Saved: {summary_path}')

# Save cosine similarities
if 'cosine_table' in dir() and cosine_table:
    cosine_path = f'{RESULTS_DIR}/cosine_vs_baseline.csv'
    with open(cosine_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['strategy'] + [f'layer_{l}' for l in layers])
        for s, sims in cosine_table.items():
            writer.writerow([s] + [f'{v:.4f}' for v in sims])
    print(f'Saved: {cosine_path}')

print('\nAll files saved to:', RESULTS_DIR)

## Interpretation Guide

| Outcome | Interpretation |
|---------|----------------|
| **All strategies ≈ baseline** | Model encodes syntactic role (subscript position), not name |
| **`misleading` drops sharply** | Model partly relies on conventional index names (i, j, key…) |
| **`all_same` holds up** | Role is fully captured by token position / context, not identity |
| **`all_same` collapses** | Token identity carries some role signal even after ambiguating names |
| **`single_chars` ≈ `random_nouns`** | Short names don't add extra confusion — same structural signal |
| **Early layers degrade more** | Lower layers rely more on surface form; higher layers capture syntax |

The **Δ F1 vs baseline** plot and the **cosine similarity** plot together reveal  
how much surface variable names contribute to the probe's learned direction.

In [ ]:
_save = RESULTS_DIR

## Multi-Model Comparison

Repeat the same probing experiment with additional models.
Results are collected in `mm_results` keyed by model nickname.

In [ ]:
# ── Additional models to evaluate ─────────────────────────────────────────
ADDITIONAL_MODELS = {
    'CodeBERT':     'microsoft/codebert-base',   # code-specific BERT, 12 layers, 768-d
    'RoBERTa':      'roberta-base',               # general NLP baseline, 12 layers, 768-d
    'Qwen2.5-0.5B': 'Qwen/Qwen2.5-0.5B',         # smaller Qwen, 24 layers, 896-d
}
print('Additional models:', list(ADDITIONAL_MODELS.keys()))


In [ ]:
from transformers import AutoTokenizer, AutoModel as _AutoModel
from collections import defaultdict as _dd
import numpy as _np
from tqdm.auto import tqdm as _tqdm

def _detect_leading_special(tok):
    probe_ids = tok('x', return_tensors='pt')['input_ids'][0].tolist()
    return len(probe_ids) - len(tok.tokenize('x'))

def _label_tokens_generic(code, target_names, tok):
    if not target_names:
        return [], []
    tokens = tok.tokenize(code)
    labels = []
    for t in tokens:
        is_cont   = t.startswith('##')
        clean     = t.lstrip('Ġ▁Ā').lstrip('##')
        clean_in  = clean.strip('[]().,;:')
        matched   = (not is_cont) and (clean in target_names or clean_in in target_names)
        labels.append(1 if matched else 0)
    return tokens, labels

def _extract_hs(dataset, tok, mdl, leading_special, max_seq=512, dev='cpu'):
    all_h  = _dd(list)
    all_l  = []
    with torch.no_grad():
        for s in _tqdm(dataset, desc='  hidden states', leave=False):
            code, labels = s['code'], s['labels']
            enc = tok(code, return_tensors='pt', truncation=True,
                      max_length=max_seq, padding=False).to(dev)
            n   = enc['input_ids'].shape[1] - leading_special
            out = mdl(**enc)

            # handle both encoder-only (hidden_states) and encoder-decoder
            hs  = out.hidden_states if hasattr(out, 'hidden_states') and out.hidden_states else None
            if hs is None:
                hs = out.encoder_hidden_states  # T5-style

            start, end = leading_special, leading_special + n
            for li, lhs in enumerate(hs):
                all_h[li].extend(lhs[0, start:end, :].float().cpu().numpy())
            all_l.extend(labels[:n])
    for li in all_h:
        all_h[li] = _np.stack(all_h[li])
    return all_h, _np.array(all_l)

def run_model_experiment(model_name, dataset, device=DEVICE):
    """Re-run the probing experiment end-to-end with a different model."""
    print(f'  Loading {model_name} ...')
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    mdl = _AutoModel.from_pretrained(model_name, output_hidden_states=True,
                                      trust_remote_code=True)
    mdl.eval().to(device)

    ls  = _detect_leading_special(tok)
    n_layers = mdl.config.num_hidden_layers
    h_size   = mdl.config.hidden_size
    print(f'  Layers={n_layers}  Hidden={h_size}  LeadingSpecial={ls}')

    # Re-label with this tokenizer (same code, same index names, different tokens)
    new_dataset = []
    for s in dataset:
        toks, labs = _label_tokens_generic(s['code'], s['target_names'], tok)
        if toks and sum(labs) > 0:
            new_dataset.append({**s, 'tokens': toks, 'labels': labs})
    print(f'  Usable samples: {len(new_dataset)}/{len(dataset)}')

    h, lbs = _extract_hs(new_dataset, tok, mdl, ls, dev=device)

    del mdl   # free memory before next model
    torch.cuda.empty_cache() if device != 'cpu' else None

    results = train_probes(h, lbs)
    best_l  = max(results, key=lambda l: results[l]['test_f1'])
    print(f'  Best layer {best_l}: acc={results[best_l]["test_acc"]:.3f}  F1={results[best_l]["test_f1"]:.3f}')
    return results, h, lbs

print('Multi-model runner defined.')


In [ ]:
# ── Run all additional models ──────────────────────────────────────────────
# Seed with Qwen2.5-1.5B results already computed above
mm_results = {'Qwen2.5-1.5B': all_results["baseline"]}
mm_hidden  = {'Qwen2.5-1.5B': all_hidden["baseline"]}
mm_labels  = {'Qwen2.5-1.5B': all_labels["baseline"]}

for nick, mname in ADDITIONAL_MODELS.items():
    print(f'\n{"="*55}')
    print(f'  Model: {nick} ({mname})')
    print(f'{"="*55}')
    try:
        res, h, lbs = run_model_experiment(mname, index_data)
        mm_results[nick] = res
        mm_hidden[nick]  = h
        mm_labels[nick]  = lbs
    except Exception as e:
        print(f'  FAILED: {e}')

print('\nModels completed:', list(mm_results.keys()))


In [ ]:
# ── Plot: test F1 per layer, all models ───────────────────────────────────
MODEL_COLORS  = ['#FF5722','#2196F3','#4CAF50','#9C27B0','#FF9800','#00BCD4']
MODEL_MARKERS = ['o','s','^','D','P','X']

layers_mm = sorted(next(iter(mm_results.values())).keys())

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, metric, ylabel in zip(axes, ['test_acc', 'test_f1'], ['Test Accuracy', 'Test Macro F1']):
    for i, (nick, res) in enumerate(mm_results.items()):
        vals = [res[l][metric] for l in layers_mm]
        ax.plot(layers_mm, vals,
                marker=MODEL_MARKERS[i % len(MODEL_MARKERS)],
                color=MODEL_COLORS[i % len(MODEL_COLORS)],
                linewidth=2, markersize=5, label=nick)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='Chance')
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f'Index/Key Probe — {ylabel}\nAll Models', fontsize=12)
    ax.set_ylim(0.3, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
_save = locals().get('RESULTS_DIR', '../results/more_perturbated_variables')
plt.savefig(f'{_save}/multimodel_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: multimodel_comparison.png')


In [ ]:
# ── Summary table: all models × best layer ────────────────────────────────
print(f'{"Model":18s}  {"Best Layer":>10}  {"Test Acc":>9}  {"Test F1":>8}')
print('-' * 52)
for nick, res in mm_results.items():
    best_l  = max(res, key=lambda l: res[l]['test_f1'])
    print(f'{nick:18s}  {best_l:>10d}  {res[best_l]["test_acc"]:>9.3f}  {res[best_l]["test_f1"]:>8.3f}')


## Cross-Model Transfer

Train probe on Model A's hidden states, evaluate directly on Model B's.
High F1 = the two models encode the index/key concept in similar directions.

In [ ]:
# ── Cross-model transfer matrix (F1) ─────────────────────────────────────
model_nicks = list(mm_results.keys())
n_models    = len(model_nicks)
xfer_matrix = _np.zeros((n_models, n_models))

for i, src_nick in enumerate(model_nicks):
    for j, tgt_nick in enumerate(model_nicks):
        if src_nick not in mm_hidden or tgt_nick not in mm_hidden:
            xfer_matrix[i, j] = float('nan')
            continue
        src_results = mm_results[src_nick]
        tgt_h       = mm_hidden[tgt_nick]
        tgt_lbs     = mm_labels[tgt_nick]

        # Use the best layer of the SOURCE model
        best_l = max(src_results, key=lambda l: src_results[l]['test_f1'])

        if best_l not in tgt_h:
            # Layer count mismatch — use the closest available layer
            available = sorted(tgt_h.keys())
            best_l    = min(available, key=lambda l: abs(l - best_l))

        probe  = src_results[best_l]['probe']
        y_pred = probe.predict(tgt_h[best_l])
        xfer_matrix[i, j] = f1_score(tgt_lbs, y_pred, average='macro')

# Print matrix
print(f'Cross-model transfer F1  (row=train model, col=test model)')
print(f'{"":18s}', end='')
for nick in model_nicks:
    print(f'  {nick[:10]:>10s}', end='')
print()
print('-' * (18 + 13 * n_models))
for i, src in enumerate(model_nicks):
    print(f'{src:18s}', end='')
    for j in range(n_models):
        v = xfer_matrix[i, j]
        print(f'  {v:>10.3f}' if not _np.isnan(v) else f'  {"N/A":>10s}', end='')
    print()


In [ ]:
# ── Heatmap: cross-model transfer ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(6, n_models * 1.8), max(5, n_models * 1.5)))
mask = _np.isnan(xfer_matrix)
im   = ax.imshow(_np.where(mask, 0, xfer_matrix), cmap='YlOrRd', vmin=0.5, vmax=1.0)

ax.set_xticks(range(n_models)); ax.set_xticklabels(model_nicks, rotation=20, ha='right', fontsize=9)
ax.set_yticks(range(n_models)); ax.set_yticklabels(model_nicks, fontsize=9)
ax.set_xlabel('Test Model',  fontsize=11)
ax.set_ylabel('Train Model', fontsize=11)
ax.set_title('Cross-Model Transfer F1\n(probe trained on row, tested on col)', fontsize=12)

for i in range(n_models):
    for j in range(n_models):
        v = xfer_matrix[i, j]
        txt = f'{v:.3f}' if not _np.isnan(v) else 'N/A'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color='black' if v < 0.8 else 'white')

plt.colorbar(im, ax=ax, label='Macro F1', fraction=0.04, pad=0.02)
plt.tight_layout()
plt.savefig(f'{_save}/crossmodel_transfer_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: crossmodel_transfer_heatmap.png')


## Cross-Perturbation Transfer Matrix

For every pair of strategies (A, B): train probe on A's hidden states, evaluate on B's.
This measures how much the learned probe direction is shared across perturbation conditions.

In [ ]:
# ── Build NxN cross-perturbation transfer matrix ──────────────────────────
strategy_list = [s for s in all_results if s in all_hidden]
n_strat       = len(strategy_list)
xfer_f1  = _np.zeros((n_strat, n_strat))
xfer_acc = _np.zeros((n_strat, n_strat))

for i, src in enumerate(strategy_list):
    for j, tgt in enumerate(strategy_list):
        src_results = all_results[src]
        tgt_h       = all_hidden[tgt]
        tgt_lbs     = all_labels[tgt]

        best_l = max(src_results, key=lambda l: src_results[l]['test_f1'])
        probe  = src_results[best_l]['probe']

        if best_l not in tgt_h:
            available = sorted(tgt_h.keys())
            best_l    = min(available, key=lambda l: abs(l - best_l))

        y_pred = probe.predict(tgt_h[best_l])
        xfer_f1[i, j]  = f1_score(tgt_lbs, y_pred, average='macro')
        xfer_acc[i, j] = accuracy_score(tgt_lbs, y_pred)

print('Cross-Perturbation Transfer F1  (row=train strategy, col=test strategy)')
strat_labels = [LABELS.get(s, s)[:22] for s in strategy_list]
print(f'{"":24s}', end='')
for sl in strat_labels:
    print(f'  {sl[:10]:>10s}', end='')
print()
print('-' * (24 + 13 * n_strat))
for i, src in enumerate(strategy_list):
    print(f'{strat_labels[i]:24s}', end='')
    for j in range(n_strat):
        print(f'  {xfer_f1[i,j]:>10.3f}', end='')
    print()


In [ ]:
# ── Heatmap: cross-perturbation F1 ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(max(10, n_strat*2.2)*2, max(7, n_strat*1.6)))

for ax, mat, title in zip(axes, [xfer_f1, xfer_acc], ['Macro F1', 'Accuracy']):
    im = ax.imshow(mat, cmap='YlOrRd', vmin=0.5, vmax=1.0)
    ax.set_xticks(range(n_strat)); ax.set_xticklabels(strat_labels, rotation=25, ha='right', fontsize=8)
    ax.set_yticks(range(n_strat)); ax.set_yticklabels(strat_labels, fontsize=8)
    ax.set_xlabel('Test Strategy', fontsize=11)
    ax.set_ylabel('Train Strategy', fontsize=11)
    ax.set_title(f'Cross-Perturbation Transfer {title}\n(probe trained on row, applied to col)', fontsize=11)
    for ii in range(n_strat):
        for jj in range(n_strat):
            ax.text(jj, ii, f'{mat[ii,jj]:.3f}', ha='center', va='center', fontsize=8,
                    color='black' if mat[ii,jj] < 0.8 else 'white')
    plt.colorbar(im, ax=ax, label=title, fraction=0.04, pad=0.02)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/cross_perturbation_transfer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cross_perturbation_transfer.png')


In [ ]:
# ── Off-diagonal drop: how much F1 falls when applied out-of-distribution ──
diag_f1 = _np.diag(xfer_f1)
print(f'{"Strategy":24s}  {"In-dist F1":>12}  {"Mean OOD F1":>13}  {"Δ drop":>8}')
print('-' * 65)
for i, s in enumerate(strategy_list):
    ood_mask = _np.ones(n_strat, dtype=bool)
    ood_mask[i] = False
    ood_mean = xfer_f1[i, ood_mask].mean()
    drop     = diag_f1[i] - ood_mean
    print(f'{strat_labels[i]:24s}  {diag_f1[i]:>12.3f}  {ood_mean:>13.3f}  {drop:>+8.3f}')
